In [1]:
!pip install -qU langchain
!pip install -qU langchain-community
!pip install -qU langchain-core
!pip install -qU langchain-huggingface
!pip install -qU faiss-cpu
!pip install -qU sentence-transformers
!pip install -qU groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.1/132.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.3/554.3 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.2 MB/s eta 0:00:00


In [3]:
import os
import numpy as np

from groq import Groq

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_core.prompts import PromptTemplate

In [13]:
import os
from google.colab import userdata
from groq import Groq

# Set API Key
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

# Initialize Groq Client
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [5]:
faq_data = [

    "What is your return policy? Returns are accepted within 30 days of delivery.",

    "How long does shipping take? Standard shipping takes 3 to 5 business days.",

    "Can I cancel an order? Orders can be cancelled before shipment.",

    "How do I track my order? Tracking details are emailed after dispatch.",

    "Do you offer international shipping? Yes, to more than 50 countries.",

    "How are refunds processed? Refunds are processed within 7 business days.",

    "What payment methods are supported? Credit cards, PayPal and UPI.",

    "How can I contact support? Email support@shop.com.",

    "Can I change my shipping address? Yes before dispatch.",

    "Do you provide discounts? Seasonal discounts are offered regularly."
]

In [6]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
vectorstore = FAISS.from_texts(
    faq_data,
    embedding_model
)

print("FAISS Index Created")

FAISS Index Created


In [8]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

In [9]:
faq_prompt = PromptTemplate(

    input_variables=["context", "question"],

    template="""
You are a FAQ assistant.

Answer ONLY using the provided FAQ context.

If the answer is not present,
say:

"I could not find that information in the FAQ database."

FAQ Context:
{context}

Question:
{question}

Answer:
"""
)

In [10]:
def retrieve_faqs(query):

    docs_scores = vectorstore.similarity_search_with_score(
        query,
        k=2
    )

    retrieved = []

    for doc, score in docs_scores:

        similarity = 1/(1+score)

        retrieved.append({
            "text": doc.page_content,
            "score": round(similarity,4)
        })

    return retrieved

In [11]:
def generate_answer(query):

    retrieved = retrieve_faqs(query)

    context = "\n\n".join(
        [item["text"] for item in retrieved]
    )

    prompt = faq_prompt.format(
        context=context,
        question=query
    )

    response = client.chat.completions.create(

        model="llama-3.3-70b-versatile",

        temperature=0,

        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ]
    )

    answer = response.choices[0].message.content

    return answer, retrieved

In [14]:
query = "How do I get a refund?"

answer, retrieved = generate_answer(query)

print("QUESTION:")
print(query)

print("\nANSWER:")
print(answer)

print("\nRETRIEVED FAQS:")

for item in retrieved:
    print(item)

QUESTION:
How do I get a refund?

ANSWER:
Refunds are processed within 7 business days.

RETRIEVED FAQS:
{'text': 'How are refunds processed? Refunds are processed within 7 business days.', 'score': np.float32(0.5371)}
{'text': 'How can I contact support? Email support@shop.com.', 'score': np.float32(0.4652)}


In [15]:
print("="*60)
print("SMART FAQ BOT")
print("Type 'exit' to quit")
print("="*60)

while True:

    query = input("\nYou: ")

    if query.lower() == "exit":
        print("\nBot: Goodbye!")
        break

    answer, retrieved = generate_answer(query)

    print("\nBot:")
    print(answer)

    print("\nTop Retrieved FAQs:")

    for idx,item in enumerate(retrieved,1):

        print(
            f"{idx}. Score={item['score']} | {item['text']}"
        )

SMART FAQ BOT
Type 'exit' to quit

You: when will i get my orde

Bot:
Standard shipping takes 3 to 5 business days.

Top Retrieved FAQs:
1. Score=0.4065000116825104 | What is your return policy? Returns are accepted within 30 days of delivery.
2. Score=0.3862000107765198 | How long does shipping take? Standard shipping takes 3 to 5 business days.

You: exit

Bot: Goodbye!
